# CIS 6211 – Foundations of Data Science
## Lab 7: Visualizing Model Performance

**Student Name:** Rana Sultan Alhinidy  
**Course:** CIS 6211 | King Khalid University


In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    mean_squared_error, mean_absolute_error, r2_score
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported successfully!")

### 📌 Cell Explanation
This cell imports all required libraries for the lab:
- **numpy, pandas** for data manipulation.
- **matplotlib, seaborn** for visualization.
- **sklearn** provides the breast cancer dataset, train/test split, and all model evaluation metrics:
  - `confusion_matrix`, `classification_report` for classification performance.
  - `roc_curve`, `auc` for the ROC curve.
  - `precision_recall_curve`, `average_precision_score` for the Precision-Recall curve.
  - `learning_curve` for plotting training vs validation performance.
  - `LogisticRegression`, `RandomForestClassifier`, `LinearRegression` as the models.

---

# Part 1: Classification Model Performance

## 1.1 Load and Prepare Data

We'll use the breast cancer dataset - a classic binary classification problem where we predict whether a tumor is malignant or benign.

In [ ]:
# Load the dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0 = malignant, 1 = benign

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts())

### 📌 Cell Explanation
The **Breast Cancer Wisconsin dataset** is loaded from sklearn. It contains 569 samples with 30 features computed from digitized images of cell nuclei.

The target variable classifies tumors as:
- **0 = Malignant** (cancerous)
- **1 = Benign** (non-cancerous)

The data is split into:
- **398 training samples** (70%) — used to fit the model
- **171 test samples** (30%) — used to evaluate performance on unseen data

The training set contains 249 benign and 149 malignant cases, showing a moderate class imbalance.

## 1.2 Train a Classification Model

In [ ]:
# Train a Logistic Regression model
clf = LogisticRegression(max_iter=10000, random_state=42)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]  # Probability of positive class

print("✅ Model trained successfully!")
print(f"Accuracy: {clf.score(X_test, y_test):.3f}")

### 📌 Cell Explanation
A **Logistic Regression** model is trained with `max_iter=10000` to ensure the optimization algorithm has enough iterations to converge.

Two types of predictions are generated:
- **`y_pred`** — hard labels (0 or 1) for direct classification
- **`y_pred_proba`** — probability estimates (between 0 and 1) needed for ROC and Precision-Recall curves

The model achieves an **accuracy of 97.7%** on the test set — excellent performance for a medical classification task.

## 1.3 Confusion Matrix

A confusion matrix shows the counts of:
- **True Positives (TP)**: Correctly predicted positive cases
- **True Negatives (TN)**: Correctly predicted negative cases
- **False Positives (FP)**: Incorrectly predicted as positive (Type I error)
- **False Negatives (FN)**: Incorrectly predicted as negative (Type II error)

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# Print detailed metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

### 📌 Cell Explanation
A **confusion matrix** is computed and visualized as a heatmap.

The matrix shows:
- **True Negatives (TN) = 61** — Malignant tumors correctly classified as malignant
- **True Positives (TP) = 106** — Benign tumors correctly classified as benign
- **False Positives (FP) = 2** — Malignant tumors incorrectly classified as benign (dangerous in medical context!)
- **False Negatives (FN) = 2** — Benign tumors incorrectly classified as malignant

**Classification Report results:**
- Precision: 0.97 (malignant), 0.98 (benign)
- Overall accuracy: **98%**

### 📝 Exercise 1.1

Based on the confusion matrix above, answer these questions:
1. How many samples were correctly classified?
2. How many false positives (benign tumors misclassified as malignant) were there?
3. Which type of error (false positive or false negative) would be more concerning in medical diagnosis? Why?

*Write your answers in the markdown cell below:*

**Your answers here:**
1.
2.
3.

### 📌 Exercise 1.1 — Answers

**1. How many samples were correctly classified?**  
167 out of 171 samples were correctly classified (61 TN + 106 TP), giving an accuracy of 97.7%.

**2. How many false positives were there?**  
There were 2 false positives — malignant tumors incorrectly classified as benign.

**3. Which type of error is more concerning in medical diagnosis? Why?**  
False Negatives (FN) are more concerning. A false negative means a malignant tumor is misclassified as benign, so the patient would not receive necessary treatment. This could allow the cancer to spread and become life-threatening. While false positives cause unnecessary worry and extra testing, they do not pose the same risk to patient safety.

## 1.4 ROC Curve (Receiver Operating Characteristic)

The ROC curve shows the trade-off between True Positive Rate (sensitivity) and False Positive Rate at different classification thresholds.

- **AUC (Area Under Curve)**: Ranges from 0 to 1. Higher is better.
  - AUC = 0.5: Random guessing
  - AUC = 1.0: Perfect classifier

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
         label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {roc_auc:.3f}")

### 📌 Cell Explanation
The **ROC (Receiver Operating Characteristic) curve** is plotted, which shows the tradeoff between True Positive Rate (Recall) and False Positive Rate at different classification thresholds.

The model achieves an **AUC of 0.998** — extremely close to perfect (1.0).

- The curve hugs the **top-left corner**, indicating excellent discrimination between malignant and benign tumors.
- The **dashed diagonal line** represents a random classifier (AUC = 0.5).

AUC interpretation: 1.0 = perfect, 0.5 = no better than random guessing.

## 1.5 Precision-Recall Curve

Precision-Recall curves are especially useful for imbalanced datasets.

- **Precision**: Of all positive predictions, how many were correct?
- **Recall**: Of all actual positives, how many did we find?

In [ ]:
# Calculate precision-recall curve
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2,
         label=f'Precision-Recall curve (AP = {avg_precision:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Average Precision Score: {avg_precision:.3f}")

### 📌 Cell Explanation
The **Precision-Recall curve** is plotted — particularly useful for imbalanced datasets.

- **Precision** = of all predicted positives, how many are truly positive? (accuracy of positive predictions)
- **Recall** = of all actual positives, how many did we catch? (coverage of true positives)

The model achieves an **Average Precision (AP) of 0.999**, meaning it maintains near-perfect precision even at very high recall levels.

The Precision-Recall curve is preferred over ROC when the dataset is heavily imbalanced.

### 📝 Exercise 1.2

Train a Random Forest classifier and compare its ROC curve with the Logistic Regression model.

*Hint: Use `RandomForestClassifier(random_state=42)` and follow the same steps as above.*

In [ ]:
# Your code here
# 1. Train a Random Forest classifier
# 2. Make predictions and get probabilities
# 3. Calculate and plot ROC curve
# 4. Compare AUC scores


---

# Part 2: Regression Model Performance

## 2.1 Generate Synthetic Regression Data

In [ ]:
# Create a synthetic regression dataset
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=5,
    n_informative=5,
    noise=10,
    random_state=42
)

# Convert to DataFrame for easier handling
X_reg = pd.DataFrame(X_reg, columns=[f'Feature_{i}' for i in range(5)])

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

print(f"Training set size: {len(X_train_reg)}")
print(f"Test set size: {len(X_test_reg)}")

### 📌 Cell Explanation
A **synthetic regression dataset** is generated with 500 samples and 5 informative features. The `noise=10` parameter adds random variation to make the problem more realistic.

The data is split into:
- **350 training samples**
- **150 test samples**

Using a synthetic dataset allows us to know the true underlying relationship and evaluate how well the model recovers it.

## 2.2 Train a Regression Model

In [ ]:
# Train a Linear Regression model
reg = LinearRegression()
reg.fit(X_train_reg, y_train_reg)

# Make predictions
y_pred_reg = reg.predict(X_test_reg)
y_train_pred_reg = reg.predict(X_train_reg)

# Calculate metrics
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("Model Performance Metrics:")
print(f"  R² Score: {r2:.3f}")
print(f"  RMSE: {rmse:.3f}")
print(f"  MAE: {mae:.3f}")

### 📌 Cell Explanation
A **Linear Regression** model is trained and evaluated using three metrics:

| Metric | Value | Meaning |
|---|---|---|
| **R² Score** | 0.992 | Model explains 99.2% of the variance — excellent fit |
| **RMSE** | 10.17 | Close to noise level (10), confirming the model captures the true relationship |
| **MAE** | 8.25 | Average prediction error of ~8.25 units |

The RMSE being close to the noise level means the model has learned the true signal and the remaining error is just random noise.

## 2.3 Prediction vs Actual Plot

This plot shows how well predictions align with actual values. Ideally, points should fall on the diagonal line.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.6, edgecolors='k')
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Predictions vs Actual Values (R² = {r2:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 📌 Cell Explanation
A scatter plot of **predicted vs actual values** is drawn.

Points cluster tightly around the **red diagonal line** (which represents perfect prediction where predicted = actual). There are no systematic deviations, meaning the model performs consistently across the full range of values.

A perfect model would have all points exactly on the diagonal line.

## 2.4 Residual Plot

Residuals are the differences between actual and predicted values. A good model should have:
- Residuals randomly scattered around zero
- No clear patterns or trends
- Constant variance (homoscedasticity)

In [ ]:
# Calculate residuals
residuals = y_test_reg - y_pred_reg

# Create residual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted Values
axes[0].scatter(y_pred_reg, residuals, alpha=0.6, edgecolors='k')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')
axes[0].grid(True, alpha=0.3)

# Histogram of Residuals
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📌 Cell Explanation
A **residual plot** and **distribution of residuals** are drawn side by side.

- **Left plot** = residuals scattered randomly around zero with no visible pattern ✅
- **Right plot** = histogram showing an approximately normal distribution of residuals ✅
- **Mean of residuals ≈ -0.94** — very close to zero, confirming no systematic bias

A good model should have residuals that are:
1. Randomly distributed around zero (no pattern)
2. Approximately normally distributed
3. With constant variance (homoscedasticity)

### 📝 Exercise 2.1

Looking at the residual plot:
1. Are the residuals randomly distributed around zero?
2. Do you see any patterns or trends?
3. What does this tell you about the model's performance?

*Write your observations below:*

**Your answers here:**
1.
2.
3.

### 📌 Exercise 2.1 — Answers

**1. Are the residuals randomly distributed around zero?**  
Yes, the residuals are randomly distributed around zero with a mean of approximately -0.94 (very close to zero). There is no systematic bias.

**2. Do you see any patterns or trends?**  
No. The residuals show constant variance (homoscedasticity) across all predicted values. The histogram confirms an approximately normal distribution.

**3. What does this tell you about the model's performance?**  
The random distribution confirms the linear regression model is a good fit. All model assumptions (linearity, constant variance, normality of errors) are satisfied. The R² score of 0.992 further confirms the model explains nearly all the variance in the data.

---

# Part 3: Learning Curves

Learning curves help diagnose:
- **Underfitting**: Both training and validation scores are low
- **Overfitting**: Large gap between training and validation scores
- **Good fit**: Both scores are high and close together

In [ ]:
def plot_learning_curve(estimator, X, y, title, cv=5):
    """
    Plot learning curves for training and validation sets.
    """
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy' if hasattr(estimator, 'predict_proba') else 'r2',
        random_state=42
    )

    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue', marker='o')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                     alpha=0.15, color='blue')

    plt.plot(train_sizes, val_mean, label='Validation score', color='red', marker='s')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                     alpha=0.15, color='red')

    plt.xlabel('Training Set Size')
    plt.ylabel('Score')
    plt.title(title)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.show()

# Example: Learning curve for Logistic Regression
plot_learning_curve(
    LogisticRegression(max_iter=10000, random_state=42),
    X_train, y_train,
    'Learning Curve - Logistic Regression'
)

### 📌 Cell Explanation
This cell defines a **learning curve function** that trains the model on increasing amounts of data (from 10% to 100% of the training set) and evaluates performance using **5-fold cross-validation**.

The plot shows:
- **Blue line** = training score
- **Red line** = validation score
- **Shaded regions** = standard deviation across folds

Learning curves help diagnose whether a model suffers from **underfitting** (both scores low) or **overfitting** (large gap between train and validation scores).

## 3.1 Comparing Different Model Complexities

In [ ]:
# Compare a simple vs complex model
models = [
    ('Simple: Decision Tree (max_depth=2)',
     DecisionTreeClassifier(max_depth=2, random_state=42)),
    ('Complex: Decision Tree (max_depth=10)',
     DecisionTreeClassifier(max_depth=10, random_state=42))
]

for name, model in models:
    plot_learning_curve(model, X_train, y_train, f'Learning Curve - {name}')

### 📌 Cell Explanation
Two Decision Tree models are compared using learning curves:

| Model | Training Score | Validation Score | Gap |
|---|---|---|---|
| Simple (max_depth=2) | ~0.95 | ~0.91 | Small ✅ |
| Complex (max_depth=10) | **1.000** | ~0.910 | **Large ⚠️** |

The **complex tree achieves a perfect training score of 1.0** — it has memorized the training data rather than learning general patterns. This is **overfitting**.

### 📝 Exercise 3.1

Based on the learning curves above:
1. Which model shows signs of overfitting? How can you tell?
2. Which model would you choose for deployment and why?
3. What could you do to improve the overfitting model?

*Write your answers below:*

**Your answers here:**
1.
2.
3.

### 📌 Exercise 3.1 — Answers

**1. Which model shows signs of overfitting? How can you tell?**  
The complex Decision Tree (max_depth=10) shows clear overfitting. Training score = 1.000 while validation score = 0.910 — a gap of 0.090. A perfect training score means the model memorized training data rather than learning generalizable patterns.

**2. Which model would you choose for deployment and why?**  
The simple Decision Tree (max_depth=2). Both models have similar validation scores (~0.91), but the simple model has a much smaller train-validation gap (0.041 vs 0.090), meaning it generalizes better. Following Occam's Razor: choose the simpler model when performance is comparable.

**3. What could you do to improve the overfitting model?**  
Options include: (1) Reduce max_depth, (2) Use min_samples_split or min_samples_leaf, (3) Apply pruning, (4) Use ensemble methods like Random Forest, (5) Add more training data, (6) Use cross-validation for hyperparameter tuning.

---

# Part 4: Feature Importance

Understanding which features are most important helps with:
- Model interpretation
- Feature selection
- Domain insights

In [ ]:
# Train a Random Forest to get feature importances
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Get feature importances
importances = rf_model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easier plotting
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'][:10], importance_df['Importance'][:10])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(importance_df.head())

### 📌 Cell Explanation
A **Random Forest with 100 trees** is trained and feature importances are extracted.

**Top 5 most important features:**
1. mean concave points (0.1419)
2. worst concave points (0.1271)
3. worst area (0.1182)
4. mean concavity (0.0881)
5. worst radius (0.0780)

The **concave points and area measurements** rank highest, which makes medical sense: malignant tumors tend to have more irregular shapes with concave regions and larger areas. Feature importance helps clinicians focus on the most diagnostically relevant measurements.

---

# Part 5: Comprehensive Model Comparison

Let's compare multiple models side-by-side using various metrics.

In [ ]:
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Define multiple models
models_to_compare = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier()
}

# Store results
results = []

# Train and evaluate each model
for name, model in models_to_compare.items():
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    accuracy = model.score(X_test, y_test)
    auc_score = roc_auc_score(y_test, y_pred_proba)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'AUC': auc_score
    })

# Create results DataFrame
results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)

print("Model Comparison Results:")
print(results_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].barh(results_df['Model'], results_df['Accuracy'], color='steelblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xlim([0.8, 1.0])

# AUC comparison
axes[1].barh(results_df['Model'], results_df['AUC'], color='coral')
axes[1].set_xlabel('AUC Score')
axes[1].set_title('Model AUC Comparison')
axes[1].set_xlim([0.8, 1.0])

plt.tight_layout()
plt.show()

### 📌 Cell Explanation
**Five classification models** are compared on the breast cancer dataset:

| Model | Accuracy | AUC |
|---|---|---|
| Logistic Regression | **97.7%** | **0.998** |
| Random Forest | 97.1% | 0.997 |
| KNN | 95.3% | 0.995 |
| SVM | 93.6% | 0.993 |
| Naive Bayes | 94.2% | 0.992 |

All models achieve AUC above 0.99, indicating the dataset is well-suited for classification. **Logistic Regression is the recommended choice**: best performance while being the simplest and most interpretable model.

### 📝 Final Exercise

Now it's your turn to apply everything you've learned!

**Task**: Choose either a classification or regression problem, train at least 2 different models, and create visualizations to compare their performance.

**Requirements**:
1. Load or generate a dataset
2. Split into train/test sets
3. Train at least 2 different models
4. Create appropriate visualizations:
   - For classification: confusion matrix, ROC curve, or precision-recall curve
   - For regression: prediction vs actual plot and residual plot
5. Compare models and explain which one you would choose

*Use the cells below for your solution:*

In [ ]:
# Your code here - Step 1: Load/generate data and split


In [ ]:
# Your code here - Step 2: Train Model 1


In [ ]:
# Your code here - Step 3: Train Model 2


In [ ]:
# Your code here - Step 4: Create visualizations


**Your model comparison and conclusion:**

*Write your analysis here - which model performed better and why?*

---

# Summary

## Key Takeaways

### Classification Metrics:
- **Confusion Matrix**: Shows true positives, true negatives, false positives, and false negatives
- **ROC Curve**: Visualizes trade-off between true positive rate and false positive rate
- **Precision-Recall Curve**: Especially useful for imbalanced datasets
- **AUC**: Single number summarizing ROC curve performance (0.5 = random, 1.0 = perfect)

### Regression Metrics:
- **Prediction vs Actual**: Points should fall on diagonal line for good predictions
- **Residual Plots**: Should show random scatter around zero with no patterns
- **R² Score**: Proportion of variance explained (0 to 1, higher is better)
- **RMSE/MAE**: Average prediction error in same units as target variable

### Model Diagnostics:
- **Learning Curves**: Identify underfitting (low scores) vs overfitting (large gap)
- **Feature Importance**: Understand which features drive predictions
- **Cross-Validation**: More robust estimate of model performance

## Best Practices
1. Always visualize your model's performance, don't just look at numbers
2. Use multiple metrics to get a complete picture
3. Check for overfitting with learning curves and validation sets
4. Consider the business context when choosing metrics (e.g., in medical diagnosis, false negatives might be more costly than false positives)
5. Compare multiple models before making a final choice

---

## Additional Resources
- [Scikit-learn Metrics Documentation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Understanding ROC Curves](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)
- [Cross-Validation Guide](https://scikit-learn.org/stable/modules/cross_validation.html)

---

**Great job completing this lab! 🎉**

You now have the tools to properly evaluate and visualize machine learning model performance!